# BoW sweep analysis

One notebook that cycles through SL, GRPO, and MaxRL sweeps. Artifacts are discovered from the canonical grids in `src/data/bag_of_words.py`; per-group traces aggregate across seeds (mean in the line, number of seeds in the hover, optional min/max error bars via `show_seed_bar=True`). Kinds whose sweep directory is missing are skipped.

In [ ]:
import polars as pl

from src import get_repo_base
from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

ARTIFACTS = get_repo_base() / "artifacts"
KINDS = [
    (
        "SL",
        BagOfWordsAnalysisConfig.from_sl_sweep,
        dict(study_base=ARTIFACTS / "bow-sl-sweep"),
    ),
    (
        "GRPO",
        BagOfWordsAnalysisConfig.from_grpo_sweep,
        dict(study_base=ARTIFACTS / "bow-grpo-sweep"),
    ),
    (
        "MaxRL (subtract-baseline)",
        BagOfWordsAnalysisConfig.from_maxrl_sweep,
        dict(study_base=ARTIFACTS / "bow-maxrl-sweep", subtract_baseline=True),
    ),
    (
        "MaxRL (no-subtract-baseline)",
        BagOfWordsAnalysisConfig.from_maxrl_sweep,
        dict(study_base=ARTIFACTS / "bow-maxrl-sweep", subtract_baseline=False),
    ),
]

epoch_axis = pl.col("epoch").alias("epoch")
PANELS_PER_EPOCH = [
    (epoch_axis, rsq_expr(split="train", y="target"), None),
    (epoch_axis, rsq_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="val", y="ground_truth"), None),
]
PANELS_BEST = [
    (rsq_expr(split="train", y="target"), corr_expr(split="train", y="ground_truth"), True),
    (corr_expr(split="train", y="target"), corr_expr(split="val", y="ground_truth"), True),
]

## Per-kind plots

For each sweep kind: one per-epoch figure (lines in `epoch`) and one best-epoch summary (one point per group at its seed-averaged optimum).

In [ ]:
for label, factory, kwargs in KINDS:
    analysis = factory(**kwargs)
    if analysis is None:
        print(f"[{label}] no artifacts at {kwargs['study_base']} \u2014 skipping")
        continue
    n_runs = sum(len(v) for v in analysis.studies.values())
    max_seeds = max(len(v) for v in analysis.studies.values())
    print(
        f"[{label}] {n_runs} runs across {len(analysis.studies)} groups "
        f"(up to {max_seeds} seeds / group)"
    )
    display(analysis.xy_plots(PANELS_PER_EPOCH, title=f"{label}: per-epoch"))
    display(analysis.xy_plots(PANELS_BEST, title=f"{label}: best-epoch"))

## Demo: `show_seed_bar=True`

Opt-in min/max-of-seeds error bars overlayed on the mean. Demonstrated on the first kind that has artifacts.

In [ ]:
for label, factory, kwargs in KINDS:
    analysis = factory(**kwargs)
    if analysis is None:
        continue
    display(
        analysis.xy_plots(
            PANELS_PER_EPOCH[:2],
            title=f"{label}: per-epoch (seed bars)",
            show_seed_bar=True,
        )
    )
    break